In [1]:
from src.module.database import oracle_import

In [2]:
marketplace_mp = oracle_import(
    """select id_, userid, p_date from toki.MONGO_MINIPROGRAMUSERLOGS partition(p_202605)
     where MINIPROGRAMID = '6821b668840548dbe178eacb'"""
)

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)


2026-08-14 15:25:14.518 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 1 min 13 sec 


In [3]:
marketplace_mp["USERID"].unique().shape

(258604,)

In [4]:
posting_url = 'https://staging-marketplace.toki.mn/ms/catalogue/v1/recommendation'

In [5]:
marketplace_mp.tail()

,ID_,USERID,P_DATE
633632,6a1c5078982236d956e8040d,65afc92e5687245adfba3f93,20260531
633633,6a1c516cf081a0899db9ed8a,66d5712c98ce1d7839f29b29,20260531
633634,6a1c516ec54f01072f707b4a,655f4c21e22c5f4c8b76d1ad,20260531
633635,6a1c51d12091a9e894b4d1b2,65d39d6735ef84b89ee1f638,20260531
633636,6a1c528e6adb6e8a91d03110,60d5415041f18afccfbdc833,20260531


In [6]:
consumer_events = oracle_import(
    "select * from toki.marketplace_consumer_EVENTS where p_date >='20260701'"
)

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)
2026-08-14 15:25:35.763 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 21 sec 72 ms


In [7]:
from datetime import datetime, timedelta

In [8]:
date_back = datetime.today().date() - timedelta(days=30)
date_back = date_back.strftime("%Y%m%d")

In [9]:
customer_activities = oracle_import(
    "select * from toki.marketplace_consumer_activities where p_date >='20260701'"
)

2026-08-14 15:28:40.496 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 3 min 4 sec 


In [10]:
customer_activities.head()

,ID_,ACTIVITYNAME,ACTIVITYDATA,CREATEDAT,UPDATEDAT,P_DATE
0,6a442142805151b025d9ff1d,cart-events,"{'cartId': '63b572a5119256f621a0c935', 'type':...",2026-07-01 04:04:18,2026-07-01 04:04:18,20260701
1,6a442142ce31add3c3475288,cart-events,"{'cartId': '5fb9457f58786d2fc4e4a6b5', 'type':...",2026-07-01 04:04:18,2026-07-01 04:04:18,20260701
2,6a442142ce31add3c347528a,cart-events,"{'cartId': '69a1cf749d7bf25a7dcdf8ad', 'type':...",2026-07-01 04:04:18,2026-07-01 04:04:18,20260701
3,6a442142ce31add3c347528c,cart-events,"{'cartId': '68d27a73ca4a9a5563b54e15', 'type':...",2026-07-01 04:04:18,2026-07-01 04:04:18,20260701
4,6a442142805151b025d9ff1f,cart-events,"{'cartId': '68d51739282d4349b9bd7681', 'type':...",2026-07-01 04:04:18,2026-07-01 04:04:18,20260701


In [11]:
customer_activities["ACTIVITYDATA"].values[0]

"{'cartId': '63b572a5119256f621a0c935', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, 'cart': {'_id': '69a1c0191449fddd6670fcc5', 'accountId': '63b572a5119256f621a0c935', 'items': [{'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, {'productId': '68febd519494859a95029a61', 'qty': 1, 'available': True, '_id': '69a54eac1449fddd66834326'}, {'productId': '68febd4c9494859a95029a58', 'qty': 1, 'available': True, '_id': '69a54eec1449fddd66834389'}], 'createdAt': '2026-02-27T16:02:33.965Z', 'updatedAt': '2026-06-30T20:04:18.220Z'}}"

In [12]:
import ast

In [13]:
ast.literal_eval(customer_activities["ACTIVITYDATA"].values[0])

{'cartId': '63b572a5119256f621a0c935',
 'type': 'PRODUCT_MODIFIED',
 'item': {'productId': '692e313c49a5eecb319a3f58',
  'qty': 1,
  'available': True,
  '_id': '69a302381449fddd66765d91'},
 'cart': {'_id': '69a1c0191449fddd6670fcc5',
  'accountId': '63b572a5119256f621a0c935',
  'items': [{'productId': '692e313c49a5eecb319a3f58',
    'qty': 1,
    'available': True,
    '_id': '69a302381449fddd66765d91'},
   {'productId': '68febd519494859a95029a61',
    'qty': 1,
    'available': True,
    '_id': '69a54eac1449fddd66834326'},
   {'productId': '68febd4c9494859a95029a58',
    'qty': 1,
    'available': True,
    '_id': '69a54eec1449fddd66834389'}],
  'createdAt': '2026-02-27T16:02:33.965Z',
  'updatedAt': '2026-06-30T20:04:18.220Z'}}

In [14]:
from datetime import datetime

snapshot_date = datetime.today().date()
snapshot_date = snapshot_date.strftime("%Y%m%d")

In [15]:
marketplace_products = oracle_import(
    f"select * from toki.marketplace_catalogue_products where SNAPSHOT_DATE >= to_date('{snapshot_date}', 'YYYYMMDD')"
)

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)
2026-08-14 15:30:53.824 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 2 min 13 sec 


In [16]:
marketplace_products.head()

,ID_,GROUPID,PRODUCTID,STOREID,SKU,BRAND,STOCK,MAINPRICE,SALEPRICE,SALEPERC,...,ASSETS,TAXONOMY,URL_,TAXON,VARIANTS,HASGIFT,CREATEDAT,UPDATEDAT,PRODUCTSTATE,SNAPSHOT_DATE
0,69aa6d6638713901eb69b744,97b7b698-5b26-57c5-b2a3-7e13674668fa,69a57bc39f4aeee4981d9c31,691aef5c2dbd3a51d50796cb,PANA-TH-55NX600KX,PANASONIC,0,2399900,2099900,12.5,...,['https://imagedelivery.net/jUCGGlEY6TCUtkJb-F...,"[{'id': '674425c2b07fff9ff4a48d4c', 'level': 0...",/69a57bc39f4aeee4981d9c31,"{'id': '674425c2b07fff9ff4a48d4c', 'label': 'З...",[],False,2026-03-06 14:00:06,2026-08-13 20:00:22,ARCHIVED,2026-08-14 00:03:49
1,69aa6d6638713901eb69b745,77b452f1-7378-5891-990c-06ee5a11c515,69a57bc39f4aeee4981d9c34,691aef5c2dbd3a51d50796cb,BSB-LED-FHD43S25,BSB,361,699900,599900,14.29,...,['https://imagedelivery.net/jUCGGlEY6TCUtkJb-F...,"[{'id': '674425c2b07fff9ff4a48d4c', 'level': 0...",/69a57bc39f4aeee4981d9c34,"{'id': '674425c2b07fff9ff4a48d4c', 'label': 'З...",[],False,2026-03-06 14:00:06,2026-08-13 14:00:21,SETTLED,2026-08-14 00:03:49
2,69aa6d6638713901eb69b746,1aba5d25-15cb-5a1e-97e5-6837d4e48850,69a57bc39f4aeee4981d9c3d,691aef5c2dbd3a51d50796cb,SHAR-2T-C42EG5MX,SHARP,1,1049900,849900,19.05,...,['https://imagedelivery.net/jUCGGlEY6TCUtkJb-F...,"[{'id': '674425c2b07fff9ff4a48d4c', 'level': 0...",/69a57bc39f4aeee4981d9c3d,"{'id': '674425c2b07fff9ff4a48d4c', 'label': 'З...",[],False,2026-03-06 14:00:06,2026-08-11 10:56:35,SETTLED,2026-08-14 00:03:49
3,69aa6d6638713901eb69b767,d4baab24-eeb6-545a-9d1a-a56950092c05,69a678e39f4aeee4981d9e6b,691aef5c2dbd3a51d50796cb,LG-83C4RLA,LG,0,16999900,13999900,17.65,...,['https://imagedelivery.net/jUCGGlEY6TCUtkJb-F...,"[{'id': '674425c2b07fff9ff4a48d4c', 'level': 0...",/69a678e39f4aeee4981d9e6b,"{'id': '674425c2b07fff9ff4a48d4c', 'label': 'З...",[],False,2026-03-06 14:00:06,2026-08-13 20:00:22,ARCHIVED,2026-08-14 00:03:49
4,69aa6d6638713901eb69b768,e5df7035-7331-5f10-a9eb-a530988ff2bd,69a678e39f4aeee4981d9e6e,691aef5c2dbd3a51d50796cb,LG-65G4RLA,LG,0,8999900,8999900,0,...,['https://imagedelivery.net/jUCGGlEY6TCUtkJb-F...,"[{'id': '674425c2b07fff9ff4a48d4c', 'level': 0...",/69a678e39f4aeee4981d9e6e,"{'id': '674425c2b07fff9ff4a48d4c', 'label': 'З...",[],False,2026-03-06 14:00:06,2026-08-13 20:00:22,ARCHIVED,2026-08-14 00:03:49


In [17]:
productid = "69fc469bab34c8d11412ec79"
marketplace_products[marketplace_products["PRODUCTID"] == productid]

,ID_,GROUPID,PRODUCTID,STOREID,SKU,BRAND,STOCK,MAINPRICE,SALEPRICE,SALEPERC,...,ASSETS,TAXONOMY,URL_,TAXON,VARIANTS,HASGIFT,CREATEDAT,UPDATEDAT,PRODUCTSTATE,SNAPSHOT_DATE
2058,6a0b3d3b38713901eb69bc30,86280094-3bd8-58e3-a377-b7f51bd157a2,69fc469bab34c8d11412ec79,6912d444458a4a2f9ca10060,SF321LF0,ROWENTA,1,199900,199900,0,...,['https://cdnp.cody.mn/spree/images/3431242/la...,"[{'id': '69fb16a36712683b9c3f5b84', 'level': 0...",/69fc469bab34c8d11412ec79,"{'id': '69fb16a36712683b9c3f5b84', 'label': 'Ү...","[{'productId': '69fc469bab34c8d11412ec79', '_i...",False,2026-05-19 00:24:26,2026-08-14 00:00:19,ARCHIVED,2026-08-14 00:03:49


In [18]:
consumer_events.shape

(572152, 11)

In [19]:
consumer_events.tail()

,ID_,EVENTNAME,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE
572147,6a7debaa4aeec3531730b932,product_click,"{'productIds': ['68e87d2f2e115cf79c95e07b', '6...",607e51e766cf4c00c5810bba,Zcw-3voYkHcKHNnh5QGOS2L9p1Qt1mqH,2026-08-13T16:07:03.467Z,Mozilla/5.0 (Linux; Android 10; STK-L21 Build/...,https://marketplace.toki.mn/home/674429ce8f734...,2026-08-14 00:07:06,2026-08-14 00:07:06,20260814
572148,6a7debb3ce31add3c36ad8e0,product_click,"{'productIds': ['68b7ee190bcb0200c3e8d7e3'], '...",5fb00a583c5f01b2fa43e50a,k0dSIrzXI96bDyoY0b0ZUmlkDpwBW8DF,2026-08-13T16:07:15.391Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like M...,https://marketplace.toki.mn/home/674429ce8f734...,2026-08-14 00:07:15,2026-08-14 00:07:15,20260814
572149,6a7debbf4229463e57069007,product_click,"{'productIds': ['67a2c1f98f15438cb882f427'], '...",607e51e766cf4c00c5810bba,Zcw-3voYkHcKHNnh5QGOS2L9p1Qt1mqH,2026-08-13T16:07:26.397Z,Mozilla/5.0 (Linux; Android 10; STK-L21 Build/...,https://marketplace.toki.mn/home/674429ce8f734...,2026-08-14 00:07:27,2026-08-14 00:07:27,20260814
572150,6a7debc08c8c31c84780c509,taxon_click,{'taxon': {'label': 'Гар утас'}},6006c89d30e91f1b3e4d6d20,KYGzRpffH6cd0FsI7ilmvlI0LgvfgttU,2026-08-13T16:07:28.093Z,Mozilla/5.0 (Linux; Android 16; SM-A556E Build...,https://marketplace.toki.mn/home/69f324685382f...,2026-08-14 00:07:28,2026-08-14 00:07:28,20260814
572151,6a7debc2ce31add3c36ad8e2,taxon_click,{'taxon': {'label': 'Компьютер'}},6006c89d30e91f1b3e4d6d20,KYGzRpffH6cd0FsI7ilmvlI0LgvfgttU,2026-08-13T16:07:30.656Z,Mozilla/5.0 (Linux; Android 16; SM-A556E Build...,https://marketplace.toki.mn/home/674429ce8f734...,2026-08-14 00:07:30,2026-08-14 00:07:30,20260814


In [20]:
consumer_events.head(2)

,ID_,EVENTNAME,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE
0,6a44ce39ce31add3c347e3d6,product_click,"{'productIds': ['69fc469bab34c8d11412ec79'], '...",66fbc5824e022311128232ae,jPAaTyDWFjD1JsHyR0ux3hewNYRvNvRy,2026-07-01T08:21:23.894Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like M...,https://marketplace.toki.mn/home/69f326e4f996d...,2026-07-01 16:22:17,2026-07-01 16:22:17,20260701
1,6a44ce3b420fe633e02e2e78,taxon_click,{'taxon': {'label': 'Гар утас'}},5ff870ee4f636263bd482270,2xI3rxpGJbBOeY4vnY_1EBSkTuAL8Pf9,2026-07-01T08:22:19.413Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_6 like M...,https://marketplace.toki.mn/home,2026-07-01 16:22:19,2026-07-01 16:22:19,20260701


In [21]:
consumer_events.groupby("EVENTNAME").count()

,ID_,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE
EVENTNAME,,,,,,,,,,
product_click,252753,252753,252753,250319,252753,252753,252753,252753,252753,252753
taxon_click,319399,310558,319399,316483,319399,319399,319399,319399,319399,319399


In [22]:
consumer_events.head(2).to_json(orient="records")

/tmp/ipykernel_1794698/4275074210.py:1: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  consumer_events.head(2).to_json(orient="records")


'[{"ID_":"6a44ce39ce31add3c347e3d6","EVENTNAME":"product_click","EVENTVALUE":"{\'productIds\': [\'69fc469bab34c8d11412ec79\'], \'taxon\': {\'label\': \'\\u04ae\\u0441\\u043d\\u0438\\u0439 \\u0445\\u044d\\u0440\\u044d\\u0433\\u0441\\u044d\\u043b\'}}","ACCOUNTID":"66fbc5824e022311128232ae","SESSIONID":"jPAaTyDWFjD1JsHyR0ux3hewNYRvNvRy","TIMESTAMP_":"2026-07-01T08:21:23.894Z","USERAGENT":"Mozilla\\/5.0 (iPhone; CPU iPhone OS 18_7 like Mac OS X) AppleWebKit\\/605.1.15 (KHTML, like Gecko) Mobile\\/15E148","URL_":"https:\\/\\/marketplace.toki.mn\\/home\\/69f326e4f996d4fdbd2ade89\\/69fb16a36712683b9c3f5b84","CREATEDAT":1782922937000,"UPDATEDAT":1782922937000,"P_DATE":"20260701"},{"ID_":"6a44ce3b420fe633e02e2e78","EVENTNAME":"taxon_click","EVENTVALUE":"{\'taxon\': {\'label\': \'\\u0413\\u0430\\u0440 \\u0443\\u0442\\u0430\\u0441\'}}","ACCOUNTID":"5ff870ee4f636263bd482270","SESSIONID":"2xI3rxpGJbBOeY4vnY_1EBSkTuAL8Pf9","TIMESTAMP_":"2026-07-01T08:22:19.413Z","USERAGENT":"Mozilla\\/5.0 (iPhone; C

In [23]:
consumer_events[consumer_events["EVENTNAME"] == "taxon_click"]["EVENTVALUE"].unique()

<StringArray>
[                                                                                                                                   '{'taxon': {'label': 'Гар утас'}}',
                                                                                                                                      '{'taxon': {'label': 'Зурагт'}}',
                                                                                                                             '{'taxon': {'label': 'Тренд технологи'}}',
                                                                                                                                   '{'taxon': {'label': 'Гал тогоо'}}',
                                                                                                                                    '{'taxon': {'label': 'Гэр ахуй'}}',
                                                                                                                                 '{'taxon': {'labe

In [24]:
consumer_events[consumer_events["ACCOUNTID"] == "6a5e47214aeec353171ccaa0"]

,ID_,EVENTNAME,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE


In [25]:
consumer_events[consumer_events["SESSIONID"] == "jPAaTyDWFjD1JsHyR0ux3hewNYRvNvRy"][
    "EVENTVALUE"
].values

<StringArray>
['{'productIds': ['69fc469bab34c8d11412ec79'], 'taxon': {'label': 'Үсний хэрэгсэл'}}',
 '{'productIds': ['6a309f8bd46aca65f808443d'], 'taxon': {'label': 'Үсний хэрэгсэл'}}',
 '{'productIds': ['6a052b7de66ad55426cacf70'], 'taxon': {'label': 'Үсний хэрэгсэл'}}',
                                                '{'taxon': {'label': 'Мик, спикер'}}',
         '{'productIds': ['69f9b680ce8b92727bb248a0'], 'taxon': {'label': 'Спикер'}}',
         '{'productIds': ['6a2bf50230ec49e38af99da7'], 'taxon': {'label': 'Спикер'}}',
                                             '{'taxon': {'label': 'Үсний хэрэгсэл'}}']
Length: 7, dtype: str

In [26]:
customer_activities["ACTIVITYDATA"].values[0]

"{'cartId': '63b572a5119256f621a0c935', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, 'cart': {'_id': '69a1c0191449fddd6670fcc5', 'accountId': '63b572a5119256f621a0c935', 'items': [{'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, {'productId': '68febd519494859a95029a61', 'qty': 1, 'available': True, '_id': '69a54eac1449fddd66834326'}, {'productId': '68febd4c9494859a95029a58', 'qty': 1, 'available': True, '_id': '69a54eec1449fddd66834389'}], 'createdAt': '2026-02-27T16:02:33.965Z', 'updatedAt': '2026-06-30T20:04:18.220Z'}}"

In [27]:
customer_activities["ACTIVITYDATA"].values[2]

"{'cartId': '69a1cf749d7bf25a7dcdf8ad', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '68d3cd51d36b9be827b44e3c', 'qty': 1, 'available': True, '_id': '6a3e98b19bd8f5a236d7ee4e'}, 'cart': {'_id': '69a1d17e1449fddd6672c3e0', 'accountId': '69a1cf749d7bf25a7dcdf8ad', 'items': [{'productId': '68d3cd51d36b9be827b44e3c', 'qty': 1, 'available': True, '_id': '6a3e98b19bd8f5a236d7ee4e'}], 'createdAt': '2026-02-27T17:16:46.492Z', 'updatedAt': '2026-06-30T20:04:18.234Z'}}"

In [28]:
customer_activities["ACTIVITYNAME"].unique()

<StringArray>
['cart-events', 'limit-events', 'order-events', 'wishlist-events']
Length: 4, dtype: str

In [29]:
customer_activities.groupby(["ACTIVITYNAME"]).count()

,ID_,ACTIVITYDATA,CREATEDAT,UPDATEDAT,P_DATE
ACTIVITYNAME,,,,,
cart-events,5223836,5223836,5223836,5223836,5223836
limit-events,102847,102847,102847,102847,102847
order-events,25206,25206,25206,25206,25206
wishlist-events,3593,3593,3593,3593,3593


In [30]:
customer_activities[customer_activities["ACTIVITYNAME"] == "cart-events"].head()[
    "ACTIVITYDATA"
].values[4]

"{'cartId': '68d51739282d4349b9bd7681', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '68d3cd51d36b9be827b44e3c', 'qty': 1, 'available': True, '_id': '6a3ab17ace94f86945bf6201'}, 'cart': {'_id': '69a250a11449fddd6672e579', 'accountId': '68d51739282d4349b9bd7681', 'items': [{'productId': '68d3cd51d36b9be827b44e3c', 'qty': 1, 'available': True, '_id': '6a3ab17ace94f86945bf6201'}], 'createdAt': '2026-02-28T02:19:13.501Z', 'updatedAt': '2026-06-30T20:04:18.250Z'}}"

In [31]:
customer_activities["ACTIVITYDATA"].values[0]

"{'cartId': '63b572a5119256f621a0c935', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, 'cart': {'_id': '69a1c0191449fddd6670fcc5', 'accountId': '63b572a5119256f621a0c935', 'items': [{'productId': '692e313c49a5eecb319a3f58', 'qty': 1, 'available': True, '_id': '69a302381449fddd66765d91'}, {'productId': '68febd519494859a95029a61', 'qty': 1, 'available': True, '_id': '69a54eac1449fddd66834326'}, {'productId': '68febd4c9494859a95029a58', 'qty': 1, 'available': True, '_id': '69a54eec1449fddd66834389'}], 'createdAt': '2026-02-27T16:02:33.965Z', 'updatedAt': '2026-06-30T20:04:18.220Z'}}"

In [32]:
# there 4 events that could potentially be used to track user activity: "view_product"
# limit-events - user checked the lease limit
# order-events - user placed an order or completed an order
# wishlist-events - user added a product to their wishlist
# card-events added card or removed card, modified in the cards etc events,

In [33]:
from src.database import pgsql_import

master_catalog_profile = pgsql_import(
    "select * from marketplace_catalog_data_extended_version3"
)

In [34]:
import pandas as pd

pd.set_option("display.max_columns", 100)

In [35]:
master_catalog_profile.head(2)

,carried_located_in,main_category,sub_category,product_category,exact_product_category,manufacturer,generic_name,actual_product,size,power_consumption,year,specifications,sku,connectivity,stock,price,dimensions,index,product_id,shop_name,discount,main_option,details,url_link,keywords,best_used_for,premium_grade,price_range,insurance,delivery,taxon_id,taxon_name,description,best_used_for_detail,taxondict,prompt_,created_at,colors,image_list,images,mainoption,data_relation_map,color,image,productstate,createdat,updatedat,productmeta,saleprice,image_urls,details_translation,group_id
0,Living Room,Electronics,Televisions,Mini LED QLED 4K TVs,Sony 85XR50 85-inch Mini LED QLED 4K HDR Googl...,SONY,Smart Television,Mini LED QLED 4K HDR Google 85 inch tv /SONY-K...,85 inches,,2026,"{""resolution"": ""UHD 4K 3840x2160p"", ""processor...",SONY-K-85XR50,"{""wifi"": ""Yes"", ""bluetooth"": ""Yes"", ""hdmi"": ""4...",2,11999900 MNT,Without stand: 1891 x 1085 x 492 mm; With stan...,shop_0_1772709420,6968a70895de95f954a9a16c,BSB Electronics,"{""regular_price"": ""12999900 MNT"", ""sale_price""...","{""screen-size"": ""85inch"", ""resolution"": ""3840x...",85-inch Sony Mini LED QLED 4K HDR Google TV wi...,https://imagedelivery.net/jUCGGlEY6TCUtkJb-Fl1...,"[""Sony TV"", ""85 inch TV"", ""Mini LED"", ""QLED"", ...",Entertainment,premium,luxury,,"Pickup, Shipping",674425c2b07fff9ff4a48d4c,tv,Screen size: 85 inch Resolution: UHD 4K 3840x2...,Optimized for home entertainment and streaming...,NaN,"{""role"": ""user"", ""content"": ""product index : 0...",2026-03-05 19:17:48.739441,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9769a9ad-9471-5347-abf4-c920d69a0e3e
1,Living Room,Electronics,Televisions,4K UHD TVs,Full Array LED 4K HDR Smart TV,Panasonic,Smart Television,Panasonic TH-75NX900,75 inch,,2026,"{""screen_type"": ""Full Array LED"", ""resolution""...",PANA-TH-75NX900M,"{""wifi"": ""Yes"", ""bluetooth"": ""Bluetooth 5.1"", ...",1,4599900 MNT,1675 x 1041 x 363 mm,shop_1_1772709420,6968a70895de95f954a9a16f,BSB Electronics,"{""regular_price"": ""5499900 MNT"", ""sale_price"":...","{""screen-size"": ""75 inch"", ""resolution"": ""3840...",PRODUCTSTATE: ARCHIVED; SYNCSTATE: SYNCED; CRE...,https://imagedelivery.net/jUCGGlEY6TCUtkJb-Fl1...,"[""Panasonic"", ""Panasonic TH-75NX900"", ""75 inch...",Entertainment,premium,high-end,,Pickup,674425c2b07fff9ff4a48d4c,tv,"Screen size: 75 inch, 1675mm Screen: Full Arra...",Ideal for home entertainment and streaming (mo...,NaN,"{""role"": ""user"", ""content"": ""product index : 1...",2026-03-05 19:18:43.058787,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,eae2fd2c-1c51-56f4-a7ec-7561dbfd11e9


In [36]:
master_catalog_profile.tail(2).to_json(orient="records")

'[{"carried_located_in":"On Person","main_category":"Computers","sub_category":"Laptops","product_category":"Consumer Laptops","exact_product_category":"Apple MacBook Pro (M5)","manufacturer":"Apple","generic_name":"Laptop","actual_product":"Apple Macbook Pro (M5)","size":"14 inch","power_consumption":"","year":"0","specifications":"{\\"chip\\": \\"M5\\", \\"storage\\": \\"1TB\\", \\"color\\": \\"Midnight Blue\\", \\"display\\": \\"Liquid Retina XDR\\", \\"refresh_rate\\": \\"120Hz\\", \\"battery_life\\": \\"18-22 hours\\", \\"ports\\": \\"MagSafe, USB-C, Thunderbolt\\", \\"security\\": \\"Touch ID\\", \\"connectivity\\": \\"Wi-Fi, Bluetooth\\"}","sku":"ANTM-3587_7E85","connectivity":"{\\"wifi\\": \\"Wi-Fi\\", \\"bluetooth\\": \\"Bluetooth\\"}","stock":"1","price":"9499000 MNT","dimensions":"","index":"shop_15_1786686022","product_id":"6a6b3cc6d338c1ddd2fdb8ea","shop_name":"ANTMALL","discount":"{}","main_option":"{\\"screen_size\\": \\"14 inch\\", \\"storage\\": \\"1TB\\", \\"color\\":

In [37]:
import pandas as pd

pd.set_option("display.max_columns", 100)

In [38]:
print(master_catalog_profile.head(2).to_json(orient="records"))

[{"carried_located_in":"Living Room","main_category":"Electronics","sub_category":"Televisions","product_category":"Mini LED QLED 4K TVs","exact_product_category":"Sony 85XR50 85-inch Mini LED QLED 4K HDR Google TV","manufacturer":"SONY","generic_name":"Smart Television","actual_product":"Mini LED QLED 4K HDR Google 85 inch tv \/SONY-K-85XR50\/","size":"85 inches","power_consumption":"","year":"2026","specifications":"{\"resolution\": \"UHD 4K 3840x2160p\", \"processor\": \"XR Processor (image enhancement AI)\", \"panel_type\": \"Mini LED QLED\", \"wide_viewing\": \"X-Wide Angle\", \"anti_reflection\": \"X-Anti Reflection\", \"operating_system\": \"ANDROID (Google TV)\", \"preinstalled_apps\": [\"Netflix\", \"Prime Video\", \"Disney+\", \"YouTube\", \"Apple TV\"], \"ai_image_enhancement\": \"Yes\", \"refresh_rate\": \"Motionflow\u2122 XR 800Hz (Native 120Hz)\", \"eye_protection\": \"Yes\", \"audio\": [\"Dolby Atmos\", \"DSEE\", \"Cinema\", \"X-Balanced Speaker\"], \"hdmi_ports\": 4, \"

## Recommendation Engine — Live API Integration

The engine is running on **port 8018** (local) and publicly via Cloudflare Tunnel.

### Available endpoint
| Endpoint | Method | Auth | Purpose |
|---|---|---|---|
| `/api/v1/events` | POST | ✓ | Ingest `customer_activities` events (order, cart, limit, wishlist, view, product_click, taxon_click) |
| `/api/v1/consumer-events` | POST | ✓ | Ingest Oracle `consumer_events` rows directly (EVENTNAME, EVENTVALUE, ACCOUNTID …) |
| `/api/v1/infer` | POST | ✓ | On-demand single-taxon inference for a user |
| `/api/v1/feed` | POST | ✓ | Multi-taxon feed — returns top-N products per taxon; auto-pushes to marketplace in background |
| `/api/v1/feed/push` | POST | ✓ | Generate feed AND synchronously push to the shop's endpoint; returns push status |
| `/api/v1/recommendations/taxon` | POST | ✓ | Taxon/category page product grid (CBF 45% + CF 30% + Pop 25%) |
| `/api/v1/recommendations/product` | POST | ✓ | Product detail page "You may also like" panel (CBF 60% + CF 40%) |
| `/api/v1/recommendations/basket` | POST | ✓ | Basket page cross-sell panel (CBF 40% + CF 35% + Pop 25%) |
| `/api/v1/health` | GET | — | Liveness check |
| `/api/v1/catalog/status` | GET | — | Catalog index stats |
| `/api/v1/metrics` | GET | — | Live ingestion/recommendation metrics snapshot |
| `/api/v1/logs/ingest` | GET | — | Recent ingest batch log |
| `/api/v1/logs/push` | GET | — | Recent marketplace push log |
| `/dashboard` | GET | — | Live HTML monitoring dashboard |

### Marketplace push payload — `POST https://staging-marketplace.toki.mn/ms/catalogue/v1/recommendation`
```json
{
  "accountId": "<account_id>",
  "products": [
    { "productId": "6a068636e452e1b264f6e6c6", "taxonId": "674425c2b07fff9ff4a48d4c" },
    { "productId": "69fc469bab34c8d11412ec79", "taxonId": "69fbef9bda75a61ceadc7607" }
  ]
}
```

```bash
curl --location 'https://staging-marketplace.toki.mn/ms/catalogue/v1/recommendation' \
  --header 'Content-Type: application/json' \
  --data '{
    "products": [
      { "productId": "6a068636e452e1b264f6e6c6", "taxonId": "674425c2b07fff9ff4a48d4c" }
    ],
    "accountId": "64b7b484fa4f99d010979ea0"
  }'
```

> **Note:** `accountId` must be a 24-char hex MongoDB ObjectId. Synthetic/test IDs are skipped by the push validator.


In [39]:
import requests

BASE = "http://localhost:8018"
HEADERS = {"Content-Type": "application/json"}

# ── Catalog status ─────────────────────────────────────────────────────────────
status = requests.get(f"{BASE}/api/v1/catalog/status").json()
print(f"Catalog: {status['catalog_size']} products | TF-IDF {status['tfidf_shape']}")
print(
    f"Taxon labels mapped: {status['taxon_label_map_size']} | Taxon slugs: {status['taxon_name_map_size']}"
)
print(
    f"Active sessions: {status['active_sessions']} | Tracked users: {status['tracked_users']}"
)


Catalog: 4817 products | TF-IDF [4817, 30000]
Taxon labels mapped: 167 | Taxon slugs: 80
Active sessions: 0 | Tracked users: 0


In [40]:
sample_consumer_rows = [
    {
        "ID_": "6a44ce39ce31add3c347e3d6",
        "EVENTNAME": "product_click",
        "EVENTVALUE": "{'productIds': ['69fc469bab34c8d11412ec79'], 'taxon': {'label': 'Үсний хэрэгсэл'}}",
        "ACCOUNTID": "66fbc5824e022311128232ae",
        "SESSIONID": "jPAaTyDWFjD1JsHyR0ux3hewNYRvNvRy",
        "TIMESTAMP_": "2026-07-01T08:21:23.894Z",
        "USERAGENT": "Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like Mac OS X) AppleWebKit/605.1.15",
    },
    {
        "ID_": "6a44ce3b420fe633e02e2e78",
        "EVENTNAME": "taxon_click",
        "EVENTVALUE": "{'taxon': {'label': 'Гар утас'}}",
        "ACCOUNTID": "5ff870ee4f636263bd482270",
        "SESSIONID": "2xI3rxpGJbBOeY4vnY_1EBSkTuAL8Pf9",
        "TIMESTAMP_": "2026-07-01T08:22:19.413Z",
        "USERAGENT": "Mozilla/5.0 (iPhone; CPU iPhone OS 18_6 like Mac OS X) AppleWebKit/605.1.15",
    },
]

r = requests.post(f"{BASE}/api/v1/consumer-events", json={"events": sample_consumer_rows})
result = r.json()
print(f"Status: {result['status']} | processed={result['processed']} failed={result['failed']}")
print()
for rec in result["recommendations"]:
    print(f"  user: {rec['id']}")
    print(f"  taxon_id: {rec['taxon_id']}")
    print(f"  strategy: {rec['strategy']} | intent: {rec['intent_score']} | device: {rec['device']}")
    print(f"  recs ({rec['count']}): {rec['recommendations'][:4]}...")
    print()


Status: accepted | processed=2 failed=0

  user: 5ff870ee4f636263bd482270
  taxon_id: None
  strategy: popular | intent: 0.5 | device: mobile
  recs (1): ['69fc469bab34c8d11412ec79']...

  user: 66fbc5824e022311128232ae
  taxon_id: 69fb16a36712683b9c3f5b84
  strategy: hybrid | intent: 1.5 | device: mobile
  recs (12): ['69fc469bab34c8d11412ec76', '69fc469bab34c8d11412ec7f', '69fc469cab34c8d11412eca2', '69fc4699ab34c8d11412ebf6']...



In [41]:
# ── POST /feed: multi-taxon recommendations ────────────────────────────────────
# account_id must be a 24-char hex MongoDB ObjectId for shop push to succeed
DEMO_ACCOUNT = "66fbc5824e022311128232ae"

requests.post(
    f"{BASE}/api/v1/events",
    json={
        "events": [
            {
                "account_id": DEMO_ACCOUNT,
                "activity_name": "product_click",
                "activity_data": {
                    "productIds": ["69fc469bab34c8d11412ec79"],
                    "taxon": {"label": "household-appliances-multi-purpose-vacuum"},
                },
            },
            {
                "account_id": DEMO_ACCOUNT,
                "activity_name": "order-events",
                "activity_data": {
                    "accountid": DEMO_ACCOUNT,
                    "productid": "698977503516dac1b3e97a6c",
                    "action": "complete",
                },
            },
            {
                "account_id": DEMO_ACCOUNT,
                "activity_name": "limit-events",
                "activity_data": {
                    "accountid": DEMO_ACCOUNT,
                    "limit_amount": 3000000,
                    "currency": "MNT",
                },
            },
            {
                "account_id": DEMO_ACCOUNT,
                "activity_name": "wishlist-events",
                "activity_data": {
                    "accountid": DEMO_ACCOUNT,
                    "productid": "6a309f8bd46aca65f8084431",
                    "action": "add",
                },
            },
        ]
    },
)

r = requests.post(
    f"{BASE}/api/v1/feed",
    json={"account_id": DEMO_ACCOUNT, "top_taxons": 3, "top_n_per_taxon": 8},
)
feed = r.json()
print(f"Strategy: {feed['strategy']} | Intent: {feed['intent_score']} | Total products: {feed['total_products']}")
print()
for tf in feed["taxon_feeds"]:
    print(f"  [{tf['taxon_name']}]")
    print(f"   taxon_id: {tf['taxon_id']}")
    print(f"   recommendations ({tf['count']}): {tf['recommendations'][:4]}...")
    print()


Strategy: multi_taxon_hybrid | Intent: 15.0 | Total products: 24

  [household-appliances-vacuum]
   taxon_id: 69fb128116b9019c3408d99a
   recommendations (8): ['6a05469d109dd372dc9bfc96', '6a02dea5e96e1eeb38ff74bf', '6a0546a1109dd372dc9bfe37', '6a02dea5e96e1eeb38ff74c2']...

  [gaming-console-accessory]
   taxon_id: 698c4044e783dbd39ed224f6
   recommendations (8): ['6a02dea6e96e1eeb38ff74f5', '6a0546a5109dd372dc9c000a', '6a0c50f2cb69d0907167363e', '6a3f830a8db83a55ca6c6956']...

  [household-appliances-multi-purpose-vacuum]
   taxon_id: 69fb126eae2e0da5c8bca3a0
   recommendations (8): ['6a054699109dd372dc9bfb34', '69fc469bab34c8d11412ec4a', '6a054699109dd372dc9bfb31', '6a0546a5109dd372dc9c005b']...



In [42]:
# ── POST /feed/push: generate feed and push to staging marketplace ─────────────
# accountId must be 24-char hex — validated before push to avoid shop 400 errors
r = requests.post(
    f"{BASE}/api/v1/feed/push",
    json={
        "account_id": DEMO_ACCOUNT,
        "top_taxons": 3,
        "top_n_per_taxon": 10,
        "shop_feed_url": posting_url,
        "push_timeout_seconds": 3.0,
    },
)
push_result = r.json()
print(f"Strategy: {push_result['strategy']} | Total products: {push_result['total_products']}")
print(f"Push status: {push_result['push_status']} | Push URL: {push_result['push_url']}")
if push_result["push_error"]:
    print(f"Push error: {push_result['push_error']}")
print()
for tf in push_result["taxon_feeds"]:
    print(f"  {tf['taxon_name'] or tf['taxon_id']}: {tf['count']} products")


Strategy: multi_taxon_hybrid | Total products: 25
Push status: ok | Push URL: https://staging-marketplace.toki.mn/ms/catalogue/v1/recommendation

  computer-gaming-pc: 10 products
  computer-gaming-pc-part-ram: 5 products
  kitchen-appliances-electric-kettle: 10 products


In [43]:
# ── Preview: exact payload that will be sent to the marketplace ────────────────
# Generates the feed locally and prints the full products list before pushing
preview_feed = requests.post(
    f"{BASE}/api/v1/feed",
    json={"account_id": DEMO_ACCOUNT, "top_taxons": 5, "top_n_per_taxon": 10},
).json()

preview_payload = {
    "accountId": DEMO_ACCOUNT,
    "products": [
        {"productId": pid, "taxonId": tf["taxon_id"]}
        for tf in preview_feed.get("taxon_feeds", [])
        for pid in tf.get("recommendations", [])
    ],
}

print(f"accountId : {preview_payload['accountId']}")
print(f"products  : {len(preview_payload['products'])} items")
print()
for i, p in enumerate(preview_payload["products"]):
    print(f"  [{i+1:>2}]  productId: {p['productId']}   taxonId: {p['taxonId']}")


accountId : 66fbc5824e022311128232ae
products  : 40 items

  [ 1]  productId: 6a05469d109dd372dc9bfc96   taxonId: 69fb128116b9019c3408d99a
  [ 2]  productId: 6a02dea5e96e1eeb38ff74bf   taxonId: 69fb128116b9019c3408d99a
  [ 3]  productId: 6a0546a1109dd372dc9bfe37   taxonId: 69fb128116b9019c3408d99a
  [ 4]  productId: 6a02dea5e96e1eeb38ff74c2   taxonId: 69fb128116b9019c3408d99a
  [ 5]  productId: 6a0546aa109dd372dc9c02bb   taxonId: 69fb128116b9019c3408d99a
  [ 6]  productId: 6a05469c109dd372dc9bfc8a   taxonId: 69fb128116b9019c3408d99a
  [ 7]  productId: 6989774f3516dac1b3e979ee   taxonId: 69fb128116b9019c3408d99a
  [ 8]  productId: 69fb11292a970a555ee2431a   taxonId: 69fb128116b9019c3408d99a
  [ 9]  productId: 6a0c50f2cb69d0907167363e   taxonId: 69fb128116b9019c3408d99a
  [10]  productId: 6a0d7de2cee06c1778ef0752   taxonId: 69fb128116b9019c3408d99a
  [11]  productId: 6a02dea6e96e1eeb38ff74f5   taxonId: 698c4044e783dbd39ed224f6
  [12]  productId: 6a0546a5109dd372dc9c000a   taxonId: 698c40

In [44]:
# ── Check ingest + push logs ───────────────────────────────────────────────────
import json

ingest_log = requests.get(f"{BASE}/api/v1/logs/ingest?limit=5").json()
push_log   = requests.get(f"{BASE}/api/v1/logs/push?limit=5").json()

print("=== Recent Ingest Batches ===")
for e in ingest_log["entries"]:
    print(f"  {e['ts'][:19]}  [{e['source']}]  processed:{e['processed']}  failed:{e['failed']}  users:{len(e['users'])}  {e['event_types']}")

print()
print("=== Recent Marketplace Pushes ===")
for e in push_log["entries"]:
    status_str = "✓" if e["push_status"] == "ok" else ("↷" if e["push_status"] == "skipped" else "✗")
    print(f"  {e['ts'][:19]}  {status_str} [{e['push_status']}]  account:{e['account_id']}  products:{e['products_count']}")
    if e.get("push_error"):
        print(f"    error: {e['push_error']}")


=== Recent Ingest Batches ===
  2026-08-14T07:30:57  [events]  processed:4  failed:0  users:1  {'product_click': 1, 'order-events': 1, 'limit-events': 1, 'wishlist-events': 1}
  2026-08-14T07:30:57  [consumer-events]  processed:2  failed:0  users:2  {'product_click': 1, 'taxon_click': 1}
  2026-08-14T07:15:59  [consumer-events]  processed:0  failed:1  users:0  {}
  2026-08-14T07:15:55  [consumer-events]  processed:0  failed:1  users:0  {}
  2026-08-14T07:15:53  [consumer-events]  processed:0  failed:1  users:0  {}

=== Recent Marketplace Pushes ===
  2026-08-14T07:30:57  ✓ [ok]  account:66fbc5824e022311128232ae  products:25
  2026-08-14T07:30:57  ✓ [ok]  account:66fbc5824e022311128232ae  products:24
  2026-08-14T07:14:15  ✓ [ok]  account:66fbc5824e022311128232ae  products:24
  2026-08-14T07:06:53  ✓ [ok]  account:f3cd98859f9d49ab91fad525  products:30
  2026-08-14T07:06:53  ✓ [ok]  account:578cfa4ab91048ecb79ad10c  products:30


In [ ]:
# ── Public URL (internet-exposed via Cloudflare Tunnel) ───────────────────────
# Share this URL with shop developers for testing
PUBLIC_URL = "http://10.22.4.13:8018"

print("=== Public API endpoints for shop developers ===")
print()
print(f"Docs/Swagger UI:       {PUBLIC_URL}/docs")
print(f"Monitoring dashboard:  {PUBLIC_URL}/dashboard")
print(f"Health check:     GET  {PUBLIC_URL}/api/v1/health")
print(f"Catalog status:   GET  {PUBLIC_URL}/api/v1/catalog/status")
print(f"Metrics:          GET  {PUBLIC_URL}/api/v1/metrics")
print()
print(f"Ingest events:    POST {PUBLIC_URL}/api/v1/events")
print(f"Consumer events:  POST {PUBLIC_URL}/api/v1/consumer-events")
print(f"Single-taxon infer:POST {PUBLIC_URL}/api/v1/infer")
print(f"Multi-taxon feed: POST {PUBLIC_URL}/api/v1/feed")
print(f"Feed + push back: POST {PUBLIC_URL}/api/v1/feed/push")
print()
print(f"Taxon page recs:  POST {PUBLIC_URL}/api/v1/recommendations/taxon")
print(f"Product page recs:POST {PUBLIC_URL}/api/v1/recommendations/product")
print(f"Basket page recs: POST {PUBLIC_URL}/api/v1/recommendations/basket")
print()

# Quick health check via public URL
try:
    r = requests.get(f"{PUBLIC_URL}/api/v1/health", timeout=10)
    print(
        f"Public health check: {r.status_code} | catalog_ready={r.json()['catalog_ready']}"
    )
except requests.exceptions.ConnectionError as e:
    print(f"Public health check: unreachable ({e})")


=== Public API endpoints for shop developers ===

Docs/Swagger UI:   http://10.22.4.13:8018/docs
Health check:      GET  http://10.22.4.13:8018/api/v1/health
Catalog status:    GET  http://10.22.4.13:8018/api/v1/catalog/status

Ingest events:     POST http://10.22.4.13:8018/api/v1/events
Consumer events:   POST http://10.22.4.13:8018/api/v1/consumer-events
Single-taxon infer:POST http://10.22.4.13:8018/api/v1/infer
Multi-taxon feed:  POST http://10.22.4.13:8018/api/v1/feed
Feed + push back:  POST http://10.22.4.13:8018/api/v1/feed/push

Public health check: 200 | catalog_ready=True


## Offline Evaluation — Recommendation Quality Pipeline

Temporal hold-out: events before `CUTOFF` are fed to the engine as training context; events after are held-out ground truth.

| Metric | What it measures |
|---|---|
| Precision@K | Fraction of top-K recommendations that were actually interacted with |
| Recall@K | Fraction of the user's test interactions recovered in top-K |
| Hit Rate@K | ≥ 1 relevant item in top-K (binary) |
| NDCG@K | Ranking quality of hits within top-K |
| MRR | Position of the first relevant item (mean reciprocal rank) |
| Coverage | % of catalog surfaced across all eval users |


In [46]:
import math
from collections import Counter

import pandas as pd

# ── Configurable evaluation parameters ────────────────────────────────────────
CUTOFF = pd.Timestamp("2026-07-22 00:00:00+00:00")  # shift to change train/test window
K_VALUES = [5, 10, 20]
MAX_EVAL_USERS = 200  # cap for tractable API evaluation
MIN_TRAIN_EVENTS = 2  # require ≥ N events in train window per user
MIN_GT_ITEMS = 1  # require ≥ N ground-truth products in test window

# ── Temporal split on consumer_events ─────────────────────────────────────────
ce = consumer_events.copy()
ce["ts"] = pd.to_datetime(ce["TIMESTAMP_"], utc=True, errors="coerce")
train_ce = ce[ce["ts"] < CUTOFF]
test_ce = ce[ce["ts"] >= CUTOFF]

print(f"Train: {train_ce['ts'].min().date()} → {train_ce['ts'].max().date()}")
print(f"Test:  {test_ce['ts'].min().date()} → {test_ce['ts'].max().date()}")
print()
print(
    f"Train events : {len(train_ce):>8,}  |  unique users: {train_ce['ACCOUNTID'].nunique():,}"
)
print(
    f"Test  events : {len(test_ce):>8,}  |  unique users: {test_ce['ACCOUNTID'].nunique():,}"
)
print(f"Users in both: {len(set(train_ce['ACCOUNTID']) & set(test_ce['ACCOUNTID'])):,}")

Train: 2026-06-28 → 2026-07-21
Test:  2026-07-22 → 2026-08-13

Train events :  287,516  |  unique users: 48,530
Test  events :  284,636  |  unique users: 44,063
Users in both: 10,540


In [47]:
import ast


def _parse_dict(val) -> dict:
    """Parse Oracle Python-dict-literal string to dict."""
    if not isinstance(val, str) or not val.strip():
        return {}
    try:
        return ast.literal_eval(val)
    except Exception:
        return {}


def _activity_product_id(row) -> str | None:
    """Extract productid from customer_activities row (order / wishlist / cart)."""
    if row.get("ACTIVITYNAME") not in (
        "order-events",
        "wishlist-events",
        "cart-events",
    ):
        return None
    try:
        d = row["ACTIVITYDATA"]
        if isinstance(d, str):
            d = ast.literal_eval(d)
        return d.get("productid") if isinstance(d, dict) else None
    except Exception:
        return None


# ── Ground truth from consumer_events: product_click → productIds ─────────────
test_pc = test_ce[test_ce["EVENTNAME"] == "product_click"].copy()
test_pc["pids"] = test_pc["EVENTVALUE"].apply(
    lambda v: [p for p in _parse_dict(v).get("productIds", []) if p] or None
)
gt_clicks: dict[str, set] = (
    test_pc.dropna(subset=["pids"])
    .explode("pids")
    .dropna(subset=["pids"])
    .groupby("ACCOUNTID")["pids"]
    .apply(set)
    .to_dict()
)
print(f"Click-based GT users (test period): {len(gt_clicks):,}")

# ── Ground truth from customer_activities: order / wishlist / cart ─────────────
gt_activities: dict[str, set] = {}
ca = customer_activities.copy()
pdate_col = next((c for c in ca.columns if c.upper() == "P_DATE"), None)
acct_col = next((c for c in ca.columns if "ACCOUNT" in c.upper()), None)
if pdate_col and acct_col:
    ca[pdate_col] = pd.to_numeric(ca[pdate_col], errors="coerce")
    cutoff_int = int(CUTOFF.strftime("%Y%m%d"))
    ca_test = ca[ca[pdate_col] >= cutoff_int].copy()
    ca_test["pid"] = ca_test.apply(_activity_product_id, axis=1)
    gt_activities = (
        ca_test.dropna(subset=["pid"]).groupby(acct_col)["pid"].apply(set).to_dict()
    )
    print(f"Activity-based GT users (test period): {len(gt_activities):,}")

# ── Merge both sources and filter by thresholds ────────────────────────────────
ground_truth: dict[str, set] = {}
for uid in set(gt_clicks) | set(gt_activities):
    pids = (gt_clicks.get(uid) or set()) | (gt_activities.get(uid) or set())
    if len(pids) >= MIN_GT_ITEMS:
        ground_truth[uid] = pids

train_event_counts = train_ce.groupby("ACCOUNTID").size()
eval_candidates = [
    u for u in ground_truth if train_event_counts.get(u, 0) >= MIN_TRAIN_EVENTS
]
# Prioritise users with richer ground truth (more test interactions = clearer signal)
eval_users_final = sorted(
    eval_candidates, key=lambda u: len(ground_truth[u]), reverse=True
)[:MAX_EVAL_USERS]

avg_gt = sum(len(ground_truth[u]) for u in eval_users_final) / max(
    1, len(eval_users_final)
)
print()
print(f"Combined GT users (≥{MIN_GT_ITEMS} test product) : {len(ground_truth):,}")
print(f"Eval candidates  (≥{MIN_TRAIN_EVENTS} train events): {len(eval_candidates):,}")
print(f"Final eval set   : {len(eval_users_final)} users  (avg GT items: {avg_gt:.1f})")

Click-based GT users (test period): 30,115

Combined GT users (≥1 test product) : 30,115
Eval candidates  (≥2 train events): 6,197
Final eval set   : 200 users  (avg GT items: 46.3)


In [48]:
# ── Warm up: ingest train-period events into the API ─────────────────────────
eval_user_set = set(eval_users_final)
train_for_eval = train_ce[train_ce["ACCOUNTID"].isin(eval_user_set)]
INGEST_BATCH = 200


def _to_consumer_row(row: dict) -> dict:
    return {
        "EVENTNAME": str(row.get("EVENTNAME", "")),
        "EVENTVALUE": str(row.get("EVENTVALUE", "")),
        "ACCOUNTID": str(row.get("ACCOUNTID", "")),
        "SESSIONID": str(row.get("SESSIONID", "")),
        "TIMESTAMP_": str(row.get("TIMESTAMP_", "")),
        "USERAGENT": str(row.get("USERAGENT", "")),
    }


rows = train_for_eval.to_dict(orient="records")
processed_total = failed_total = 0

for i in range(0, len(rows), INGEST_BATCH):
    chunk = [_to_consumer_row(r) for r in rows[i : i + INGEST_BATCH]]
    resp = requests.post(f"{BASE}/api/v1/consumer-events", json={"events": chunk}, timeout=30)
    if resp.status_code == 200:
        d = resp.json()
        processed_total += d.get("processed", 0)
        failed_total += d.get("failed", 0)
    else:
        failed_total += len(chunk)

after = requests.get(f"{BASE}/api/v1/catalog/status").json()
print(f"Train events — processed: {processed_total:,}  failed: {failed_total:,}")
print(f"API tracked users after warm-up: {after['tracked_users']:,}")


Train events — processed: 5,826  failed: 56
API tracked users after warm-up: 130


In [49]:
def _precision_at_k(recs: list, relevant: set, k: int) -> float:
    return sum(1 for p in recs[:k] if p in relevant) / k if k else 0.0


def _recall_at_k(recs: list, relevant: set, k: int) -> float:
    if not relevant:
        return 0.0
    return sum(1 for p in recs[:k] if p in relevant) / len(relevant)


def _hit_rate_at_k(recs: list, relevant: set, k: int) -> float:
    return float(any(p in relevant for p in recs[:k]))


def _ndcg_at_k(recs: list, relevant: set, k: int) -> float:
    dcg = sum(1.0 / math.log2(i + 2) for i, p in enumerate(recs[:k]) if p in relevant)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg else 0.0


def _mrr(recs: list, relevant: set) -> float:
    for i, p in enumerate(recs):
        if p in relevant:
            return 1.0 / (i + 1)
    return 0.0


# ── Generate feed + score against ground truth ────────────────────────────────
eval_records = []
all_recommended_pids: set = set()
strategy_counter: Counter = Counter()

for uid in eval_users_final:
    relevant = ground_truth[uid]
    resp = requests.post(
        f"{BASE}/api/v1/feed",
        json={"account_id": uid, "top_taxons": 5, "top_n_per_taxon": 12},
        timeout=15,
    )
    if resp.status_code != 200:
        continue

    data = resp.json()
    strategy_counter[data.get("strategy", "unknown")] += 1

    flat_recs = [
        pid
        for tf in data.get("taxon_feeds", [])
        for pid in tf.get("recommendations", [])
    ]
    all_recommended_pids.update(flat_recs)
    if not flat_recs:
        continue

    row: dict = {
        "strategy": data.get("strategy"),
        "intent_score": data.get("intent_score", 0.0),
        "n_recs": len(flat_recs),
        "n_relevant": len(relevant),
        "mrr": _mrr(flat_recs, relevant),
    }
    for k in K_VALUES:
        row[f"prec@{k}"] = _precision_at_k(flat_recs, relevant, k)
        row[f"rec@{k}"] = _recall_at_k(flat_recs, relevant, k)
        row[f"hr@{k}"] = _hit_rate_at_k(flat_recs, relevant, k)
        row[f"ndcg@{k}"] = _ndcg_at_k(flat_recs, relevant, k)
    eval_records.append(row)

eval_df = pd.DataFrame(eval_records)
pct = len(eval_df) / max(1, len(eval_users_final)) * 100
print(f"Evaluated {len(eval_df)} / {len(eval_users_final)} users  ({pct:.0f}%)")


Evaluated 200 / 200 users  (100%)


In [50]:
# ── Metrics summary ────────────────────────────────────────────────────────────
if eval_df.empty:
    print("No evaluation results. Re-run warm-up and evaluate cells.")
else:
    metric_cols = [
        f"{m}@{k}" for m in ("prec", "rec", "hr", "ndcg") for k in K_VALUES
    ] + ["mrr"]
    means = eval_df[metric_cols].mean()

    W = 9
    header = f"{'':13}" + "".join(f"{'K='+str(k):>{W}}" for k in K_VALUES)
    sep = "=" * len(header)
    print(sep)
    print(f"  Offline eval — {len(eval_df)} users | cutoff {CUTOFF.date()}")
    print(sep)
    print(header)
    print("-" * len(header))
    for prefix, label in [
        ("prec", "Precision"),
        ("rec", "Recall"),
        ("hr", "Hit Rate"),
        ("ndcg", "NDCG"),
    ]:
        vals = "".join(f"{means[f'{prefix}@{k}']:{W}.4f}" for k in K_VALUES)
        print(f"{label:<13}{vals}")
    print(f"{'MRR':<13}{means['mrr']:{W}.4f}")
    print()

    # Catalog coverage
    cat_n = status.get("catalog_size", 1)
    print(
        f"Catalog coverage : {len(all_recommended_pids)/cat_n:.2%}  ({len(all_recommended_pids):,} unique / {cat_n:,} total)"
    )
    print(f"Avg recs/user    : {eval_df['n_recs'].mean():.1f}")
    print(f"Avg GT items/user: {eval_df['n_relevant'].mean():.1f}")
    print()

    # Strategy breakdown
    print("Strategy distribution:")
    for strat, cnt in strategy_counter.most_common():
        bar = "█" * int(cnt / max(strategy_counter.values()) * 30)
        print(f"  {strat:<38} {cnt:>4}  ({cnt/len(eval_df)*100:5.1f}%)  {bar}")
    print()

    # Quality thresholds from PLAN.md
    print("Quality targets (PLAN.md):")
    for metric, thr in [
        ("rec@10", 0.25),
        ("ndcg@10", 0.18),
        ("hr@5", 0.40),
        ("mrr", 0.20),
    ]:
        val = means.get(metric, 0.0)
        print(
            f"  {'✓' if val >= thr else '✗'} {metric:<10} {val:.4f}  (target ≥ {thr})"
        )
    print()

    # Per-strategy breakdown of hit rate
    if "strategy" in eval_df.columns:
        print("Hit Rate@10 by strategy:")
        strat_hr = (
            eval_df.groupby("strategy")["hr@10"]
            .agg(["mean", "count"])
            .sort_values("mean", ascending=False)
        )
        for strat, row_s in strat_hr.iterrows():
            print(f"  {strat:<38} {row_s['mean']:.4f}  (n={int(row_s['count'])})")

  Offline eval — 200 users | cutoff 2026-07-22
                   K=5     K=10     K=20
----------------------------------------
Precision       0.1090   0.0635   0.0405
Recall          0.0108   0.0127   0.0162
Hit Rate        0.2200   0.2700   0.3100
NDCG            0.1123   0.0791   0.0572
MRR             0.1704

Catalog coverage : 32.36%  (1,559 unique / 4,817 total)
Avg recs/user    : 37.3
Avg GT items/user: 46.3

Strategy distribution:
  multi_taxon_hybrid                      200  (100.0%)  ██████████████████████████████

Quality targets (PLAN.md):
  ✗ rec@10     0.0127  (target ≥ 0.25)
  ✗ ndcg@10    0.0791  (target ≥ 0.18)
  ✗ hr@5       0.2200  (target ≥ 0.4)
  ✗ mrr        0.1704  (target ≥ 0.2)

Hit Rate@10 by strategy:
  multi_taxon_hybrid                     0.2700  (n=200)


### Multi-Cutoff Sweep — Model Stability Across Timeline Ranges

Re-runs the evaluation across several cutoff dates without re-ingesting events.
Confirms the model performs consistently as the train window shrinks.


In [51]:
SWEEP_CUTOFFS = [
    pd.Timestamp("2026-07-10 00:00:00+00:00"),
    pd.Timestamp("2026-07-17 00:00:00+00:00"),
    pd.Timestamp("2026-07-22 00:00:00+00:00"),
    pd.Timestamp("2026-07-28 00:00:00+00:00"),
]
SWEEP_K = 10
SWEEP_MAX_USERS = 50

sweep_rows = []
for cutoff_ts in SWEEP_CUTOFFS:
    t_ce = ce[ce["ts"] < cutoff_ts]
    e_ce = ce[ce["ts"] >= cutoff_ts]

    e_pc = e_ce[e_ce["EVENTNAME"] == "product_click"].copy()
    e_pc["pids"] = e_pc["EVENTVALUE"].apply(
        lambda v: [p for p in _parse_dict(v).get("productIds", []) if p] or None
    )
    gt_c: dict[str, set] = (
        e_pc.dropna(subset=["pids"])
        .explode("pids")
        .dropna(subset=["pids"])
        .groupby("ACCOUNTID")["pids"]
        .apply(set)
        .to_dict()
    )
    t_counts = t_ce.groupby("ACCOUNTID").size()
    cands = [u for u in gt_c if t_counts.get(u, 0) >= MIN_TRAIN_EVENTS]
    users = sorted(cands, key=lambda u: len(gt_c[u]), reverse=True)[:SWEEP_MAX_USERS]

    hits, ndcgs = [], []
    for uid in users:
        rel = gt_c[uid]
        rsp = requests.post(
            f"{BASE}/api/v1/feed",
            json={"account_id": uid, "top_taxons": 5, "top_n_per_taxon": 12},
            timeout=15,
        )
        if rsp.status_code != 200:
            continue
        recs = [
            p
            for tf in rsp.json().get("taxon_feeds", [])
            for p in tf.get("recommendations", [])
        ]
        if not recs:
            continue
        hits.append(_hit_rate_at_k(recs, rel, SWEEP_K))
        ndcgs.append(_ndcg_at_k(recs, rel, SWEEP_K))

    if hits:
        sweep_rows.append(
            {
                "cutoff": str(cutoff_ts.date()),
                "train_days": (cutoff_ts - ce["ts"].min()).days,
                "test_days": (ce["ts"].max() - cutoff_ts).days,
                "eval_users": len(hits),
                f"hr@{SWEEP_K}": sum(hits) / len(hits),
                f"ndcg@{SWEEP_K}": sum(ndcgs) / len(ndcgs),
            }
        )

sweep_df = pd.DataFrame(sweep_rows)
print("Multi-cutoff sweep results:")
print(sweep_df.set_index("cutoff").round(4).to_string())


Multi-cutoff sweep results:
            train_days  test_days  eval_users  hr@10  ndcg@10
cutoff                                                       
2026-07-10          11         34          50   0.50   0.1738
2026-07-17          18         27          50   0.58   0.2370
2026-07-22          23         22          50   0.58   0.2205
2026-07-28          29         16          50   0.56   0.2012


In [52]:
# ── Fix 1: rebuild ground truth including customer_activities ─────────────────
# The original code used acct_col (None) and _activity_product_id which both
# looked for flat keys like 'accountid' / 'productid'. The actual ACTIVITYDATA
# nests them: accountId → cart.accountId, productId → item.productId.

cutoff_int = int(CUTOFF.strftime("%Y%m%d"))


def _nested_get(d: dict, *paths) -> str | None:
    """Try each key path (tuple = nested, str = flat) and return first match."""
    for path in paths:
        keys = (path,) if isinstance(path, str) else path
        v = d
        for k in keys:
            if not isinstance(v, dict):
                v = None
                break
            # case-insensitive lookup
            v = v.get(k) or next((v[dk] for dk in v if dk.lower() == k.lower()), None)
        if v and isinstance(v, str):
            return v
    return None


def _extract_aid_fixed(row) -> str | None:
    try:
        d = _parse_dict(row["ACTIVITYDATA"])
        return _nested_get(
            d,
            ("cart", "accountId"),  # cart-events
            "accountId", "accountid", "account_id",  # other formats
        )
    except Exception:
        return None


def _extract_pid_fixed(row) -> str | None:
    if row.get("ACTIVITYNAME") not in ("order-events", "wishlist-events", "cart-events"):
        return None
    try:
        d = _parse_dict(row["ACTIVITYDATA"])
        return _nested_get(
            d,
            ("item", "productId"),  # cart-events nested item
            "productId", "productid", "product_id",
        )
    except Exception:
        return None


ca_fixed = ca.copy()
if pdate_col:
    ca_fixed[pdate_col] = pd.to_numeric(ca_fixed[pdate_col], errors="coerce")
ca_fixed["_aid"] = ca_fixed.apply(_extract_aid_fixed, axis=1)
ca_fixed["_pid"] = ca_fixed.apply(_extract_pid_fixed, axis=1)

ca_test_fixed = ca_fixed[ca_fixed[pdate_col] >= cutoff_int] if pdate_col else ca_fixed
gt_activities_fixed: dict[str, set] = (
    ca_test_fixed.dropna(subset=["_aid", "_pid"])
    .groupby("_aid")["_pid"]
    .apply(set)
    .to_dict()
)
print(f"gt_activities (fixed) : {len(gt_activities_fixed):,} users")

# Merge click-based + activity-based ground truth
ground_truth2: dict[str, set] = {}
for uid in set(gt_clicks) | set(gt_activities_fixed):
    pids = (gt_clicks.get(uid) or set()) | (gt_activities_fixed.get(uid) or set())
    if len(pids) >= MIN_GT_ITEMS:
        ground_truth2[uid] = pids

eval_candidates2 = [u for u in ground_truth2 if train_event_counts.get(u, 0) >= MIN_TRAIN_EVENTS]
eval_users2 = sorted(eval_candidates2, key=lambda u: len(ground_truth2[u]), reverse=True)[:MAX_EVAL_USERS]
avg_gt2 = sum(len(ground_truth2[u]) for u in eval_users2) / max(1, len(eval_users2))

print(f"Combined GT users    : {len(ground_truth2):,}")
print(f"Final eval set       : {len(eval_users2)} users  (avg GT items: {avg_gt2:.1f})")
print(f"vs original          : {len(eval_users_final)} users  (avg GT: {avg_gt:.1f})")
print(f"Max achievable Rec@10: {10 / max(1, avg_gt2):.4f}  (10 / avg_gt)")


gt_activities (fixed) : 48,371 users
Combined GT users    : 67,296
Final eval set       : 200 users  (avg GT items: 46.8)
vs original          : 200 users  (avg GT: 46.3)
Max achievable Rec@10: 0.2139  (10 / avg_gt)


In [53]:
# ── Fix 2: also warm up with customer_activities (order / cart / wishlist) ─────
ca_train_fixed = ca_fixed.copy()
if pdate_col:
    ca_train_fixed = ca_train_fixed[ca_train_fixed[pdate_col] < cutoff_int]

ca_eval_train = ca_train_fixed[
    ca_train_fixed["_aid"].isin(eval_user_set)
    & ca_train_fixed["ACTIVITYNAME"].isin(
        ["order-events", "cart-events", "wishlist-events", "view_product"]
    )
].copy()

print(f"customer_activities rows for eval users (train): {len(ca_eval_train):,}")
if not ca_eval_train.empty:
    print(ca_eval_train["ACTIVITYNAME"].value_counts().to_string())

ca_processed = ca_failed = 0
for i in range(0, len(ca_eval_train), INGEST_BATCH):
    events = []
    for r in ca_eval_train.iloc[i : i + INGEST_BATCH].to_dict(orient="records"):
        aid = r.get("_aid", "")
        if not aid:
            continue
        events.append(
            {
                "account_id": str(aid),
                "activity_name": r.get("ACTIVITYNAME", ""),
                "activity_data": r.get("ACTIVITYDATA", ""),
            }
        )
    if not events:
        continue
    resp = requests.post(f"{BASE}/api/v1/events", json={"events": events}, timeout=30)
    if resp.status_code == 200:
        d = resp.json()
        ca_processed += d.get("processed", 0)
        ca_failed += d.get("failed", 0)
    else:
        ca_failed += len(events)

status_after2 = requests.get(f"{BASE}/api/v1/catalog/status").json()
print(f"\ncustomer_activities ingestion — processed: {ca_processed:,}  failed: {ca_failed:,}")
print(
    f"Tracked users now : {status_after2['tracked_users']:,}"
    f"  (was {after['tracked_users']:,}"
    f"  Δ +{status_after2['tracked_users'] - after['tracked_users']})"
)


customer_activities rows for eval users (train): 3,715
ACTIVITYNAME
cart-events        3664
wishlist-events      51

customer_activities ingestion — processed: 3,715  failed: 0
Tracked users now : 187  (was 130  Δ +57)


In [54]:
eval_records2 = []
all_pids2: set = set()
strategy_counter2: Counter = Counter()

for uid in eval_users2:
    relevant = ground_truth2[uid]
    try:
        resp = requests.post(
            f"{BASE}/api/v1/feed",
            json={"account_id": uid, "top_taxons": 5, "top_n_per_taxon": 20},
            timeout=15,
        )
        resp.raise_for_status()
        data = resp.json()
    except (requests.RequestException, ValueError):
        continue
    strategy_counter2[data.get("strategy", "unknown")] += 1
    flat_recs = [p for tf in data.get("taxon_feeds", []) for p in tf.get("recommendations", [])]
    all_pids2.update(flat_recs)
    if not flat_recs:
        continue
    row = {
        "strategy": data.get("strategy"),
        "intent_score": data.get("intent_score", 0.0),
        "n_recs": len(flat_recs),
        "n_relevant": len(relevant),
        "mrr": _mrr(flat_recs, relevant),
    }
    for k in K_VALUES:
        row[f"prec@{k}"] = _precision_at_k(flat_recs, relevant, k)
        row[f"rec@{k}"]  = _recall_at_k(flat_recs, relevant, k)
        row[f"hr@{k}"]   = _hit_rate_at_k(flat_recs, relevant, k)
        row[f"ndcg@{k}"] = _ndcg_at_k(flat_recs, relevant, k)
    eval_records2.append(row)

eval_df2 = pd.DataFrame(eval_records2)
if eval_df2.empty:
    print("No evaluation results collected — check API connectivity and eval_users2.")
else:
    means2 = eval_df2[metric_cols].mean()
    means1 = eval_df[metric_cols].mean()

    W = 9
    header = f"{'':13}" + "".join(f"{'K='+str(k):>{W}}" for k in K_VALUES)
    sep = "=" * len(header)
    print(sep)
    print(f"  Improved eval — {len(eval_df2)} users | GT: click + order/cart/wishlist")
    print(sep)
    print(header)
    print("-" * len(header))
    for prefix, label in [("prec","Precision"),("rec","Recall"),("hr","Hit Rate"),("ndcg","NDCG")]:
        old = "".join(f"{means1[f'{prefix}@{k}']:{W}.4f}" for k in K_VALUES)
        new = "".join(f"{means2[f'{prefix}@{k}']:{W}.4f}" for k in K_VALUES)
        delta = "".join(
            f"{(means2[f'{prefix}@{k}']-means1[f'{prefix}@{k}']):+{W}.4f}" for k in K_VALUES
        )
        print(f"{label:<13}{new}  ← new")
        print(f"{'(baseline)':<13}{old}")
        print(f"{'(Δ)':<13}{delta}")
        print()
    print(f"{'MRR':<13}{means2['mrr']:{W}.4f}  (baseline: {means1['mrr']:.4f}  Δ{means2['mrr']-means1['mrr']:+.4f})")
    print()
    cat_n = status.get("catalog_size", 1)
    max_rec10 = 10 / max(1, avg_gt2)
    print(f"Catalog coverage  : {len(all_pids2)/cat_n:.2%}  ({len(all_pids2):,}/{cat_n:,})")
    print(f"Avg recs / user   : {eval_df2['n_recs'].mean():.1f}")
    print(f"Avg GT items/user : {eval_df2['n_relevant'].mean():.1f}")
    print(f"Max achievable Rec@10: {max_rec10:.4f}  (10 / avg_gt)")
    print()
    print("Quality targets (PLAN.md):")
    for metric, thr in [("rec@10", 0.25), ("ndcg@10", 0.18), ("hr@5", 0.40), ("mrr", 0.20)]:
        val = means2.get(metric, 0.0)
        impossible = (metric == "rec@10" and max_rec10 < thr)
        flag = "✓" if val >= thr else ("✗ (impossible)" if impossible else "✗")
        print(f"  {flag} {metric:<10} {val:.4f}  (target ≥ {thr})")
    print()
    print("Strategy distribution:")
    for strat, cnt in strategy_counter2.most_common():
        bar = "█" * int(cnt / max(strategy_counter2.values()) * 30)
        print(f"  {strat:<38} {cnt:>4}  ({cnt/max(1,len(eval_df2))*100:5.1f}%)  {bar}")


  Improved eval — 200 users | GT: click + order/cart/wishlist
                   K=5     K=10     K=20
----------------------------------------
Precision       0.2050   0.1125   0.0720  ← new
(baseline)      0.1090   0.0635   0.0405
(Δ)            +0.0960  +0.0490  +0.0315

Recall          0.0217   0.0238   0.0309  ← new
(baseline)      0.0108   0.0127   0.0162
(Δ)            +0.0109  +0.0112  +0.0146

Hit Rate        0.4400   0.4650   0.5200  ← new
(baseline)      0.2200   0.2700   0.3100
(Δ)            +0.2200  +0.1950  +0.2100

NDCG            0.2142   0.1460   0.1053  ← new
(baseline)      0.1123   0.0791   0.0572
(Δ)            +0.1019  +0.0669  +0.0482

MRR             0.3352  (baseline: 0.1704  Δ+0.1648)

Catalog coverage  : 41.23%  (1,986/4,817)
Avg recs / user   : 38.1
Avg GT items/user : 46.8
Max achievable Rec@10: 0.2139  (10 / avg_gt)

Quality targets (PLAN.md):
  ✗ (impossible) rec@10     0.0238  (target ≥ 0.25)
  ✗ ndcg@10    0.1460  (target ≥ 0.18)
  ✓ hr@5       0.4400 

In [55]:
eval_records3 = []
all_pids3: set = set()
strategy_counter3: Counter = Counter()

for uid in eval_users_final:
    relevant = ground_truth[uid]
    resp = requests.post(
        f"{BASE}/api/v1/feed",
        json={"account_id": uid, "top_taxons": 5, "top_n_per_taxon": 20},
        timeout=15,
    )
    if resp.status_code != 200:
        continue
    data = resp.json()
    strategy_counter3[data.get("strategy", "unknown")] += 1
    flat_recs = [p for tf in data.get("taxon_feeds", []) for p in tf.get("recommendations", [])]
    all_pids3.update(flat_recs)
    if not flat_recs:
        continue
    row = {
        "strategy": data.get("strategy"),
        "intent_score": data.get("intent_score", 0.0),
        "n_recs": len(flat_recs),
        "n_relevant": len(relevant),
        "mrr": _mrr(flat_recs, relevant),
    }
    for k in K_VALUES:
        row[f"prec@{k}"] = _precision_at_k(flat_recs, relevant, k)
        row[f"rec@{k}"]  = _recall_at_k(flat_recs, relevant, k)
        row[f"hr@{k}"]   = _hit_rate_at_k(flat_recs, relevant, k)
        row[f"ndcg@{k}"] = _ndcg_at_k(flat_recs, relevant, k)
    eval_records3.append(row)

eval_df3 = pd.DataFrame(eval_records3)
means3 = eval_df3[metric_cols].mean()
means1 = eval_df[metric_cols].mean()

W = 9
header = f"{'':13}" + "".join(f"{'K='+str(k):>{W}}" for k in K_VALUES)
sep = "=" * len(header)
print(sep)
print(f"  Click GT / enriched profiles — {len(eval_df3)} users")
print(f"  Tracked: 101 → 162  (+{162-101} users now have product history)")
print(sep)
print(header)
print("-" * len(header))
for prefix, label in [("prec","Precision"),("rec","Recall"),("hr","Hit Rate"),("ndcg","NDCG")]:
    old   = "".join(f"{means1[f'{prefix}@{k}']:{W}.4f}" for k in K_VALUES)
    new   = "".join(f"{means3[f'{prefix}@{k}']:{W}.4f}" for k in K_VALUES)
    delta = "".join(f"{(means3[f'{prefix}@{k}']-means1[f'{prefix}@{k}']):+{W}.4f}" for k in K_VALUES)
    print(f"{label:<13}{new}  ← enriched")
    print(f"{'(baseline)':<13}{old}")
    print(f"{'(Δ)':<13}{delta}")
    print()
print(f"{'MRR':<13}{means3['mrr']:{W}.4f}  (baseline: {means1['mrr']:.4f}  Δ{means3['mrr']-means1['mrr']:+.4f})")
print()
print("Quality targets (PLAN.md)  [note: rec@10 max possible = 0.2216]:")
for metric, thr in [("rec@10", 0.25), ("ndcg@10", 0.18), ("hr@5", 0.40), ("mrr", 0.20)]:
    val = means3.get(metric, 0.0)
    impossible = (metric == "rec@10" and (10 / max(1, avg_gt)) < thr)
    flag = "✓" if val >= thr else ("✗ (impossible)" if impossible else "✗")
    print(f"  {flag} {metric:<10} {val:.4f}  (target ≥ {thr})")
print()
print("Strategy breakdown (enriched profiles):")
for strat, cnt in strategy_counter3.most_common():
    bar = "█" * int(cnt / max(strategy_counter3.values()) * 30)
    print(f"  {strat:<38} {cnt:>4}  ({cnt/max(1,len(eval_df3))*100:5.1f}%)  {bar}")


  Click GT / enriched profiles — 200 users
  Tracked: 101 → 162  (+61 users now have product history)
                   K=5     K=10     K=20
----------------------------------------
Precision       0.2360   0.1330   0.0848  ← enriched
(baseline)      0.1090   0.0635   0.0405
(Δ)            +0.1270  +0.0695  +0.0443

Recall          0.0261   0.0292   0.0383  ← enriched
(baseline)      0.0108   0.0127   0.0162
(Δ)            +0.0153  +0.0165  +0.0221

Hit Rate        0.5100   0.5450   0.6150  ← enriched
(baseline)      0.2200   0.2700   0.3100
(Δ)            +0.2900  +0.2750  +0.3050

NDCG            0.2435   0.1687   0.1218  ← enriched
(baseline)      0.1123   0.0791   0.0572
(Δ)            +0.1312  +0.0896  +0.0646

MRR             0.3784  (baseline: 0.1704  Δ+0.2079)

Quality targets (PLAN.md)  [note: rec@10 max possible = 0.2216]:
  ✗ (impossible) rec@10     0.0292  (target ≥ 0.25)
  ✗ ndcg@10    0.1687  (target ≥ 0.18)
  ✓ hr@5       0.5100  (target ≥ 0.4)
  ✓ mrr        0.3784  (

In [ ]:
import time

# ── Verify: events → auto-push to marketplace ─────────────────────────────────
# Send events for a real account, then confirm the push log shows a new entry
VERIFY_ACCOUNT = "66fbc5824e022311128232ae"

push_before = requests.get(f"{BASE}/api/v1/logs/push?limit=50").json()
ts_before = {e["ts"] for e in push_before["entries"] if e.get("account_id") == VERIFY_ACCOUNT}

r = requests.post(
    f"{BASE}/api/v1/events",
    json={
        "events": [
            {
                "account_id": VERIFY_ACCOUNT,
                "activity_name": "product_click",
                "activity_data": {"productIds": ["69fc469bab34c8d11412ec79"]},
            }
        ]
    },
)
print(f"Ingest response: {r.json()['status']}  processed={r.json()['processed']}")

# Background push is fire-and-forget — give it a moment
time.sleep(3)

push_after = requests.get(f"{BASE}/api/v1/logs/push?limit=50").json()
new_entries = [
    e for e in push_after["entries"]
    if e.get("account_id") == VERIFY_ACCOUNT and e["ts"] not in ts_before
]

if new_entries:
    e = new_entries[0]
    status_sym = "✓" if e["push_status"] == "ok" else "✗"
    print(f"\n{status_sym} Marketplace push triggered automatically:")
    print(f"  account  : {e['account_id']}")
    print(f"  products : {e['products_count']}")
    print(f"  strategy : {e.get('strategy', '—')}")
    print(f"  status   : {e['push_status']}")
    print(f"  url      : {e.get('push_url', '—')}")
    if e.get("push_error"):
        print(f"  error    : {e['push_error']}")
else:
    print("\n⚠ No new push entry found — check MARKETPLACE_API_BASE_URL in config.py")
    print(f"  Push log total: {push_after['total_stored']}")


---

## Stress Test — Load Report

**Configuration:** 50 concurrent users · spawn rate 10/s · 60-second window  
**Engine:** TOKI Marketplace Recommendation System v2 · gunicorn 4× UvicornWorker · port 8018  
**Catalog:** 4,207 products · 75 taxons · TF-IDF [4207 × 30,000]  
**Seeded IDs:** 430 real product IDs · 38 real taxon IDs (harvested from live feed before test)

---

### Per-endpoint results

| Endpoint | Requests | p50 | p95 | p99 | Failures |
|---|---:|---:|---:|---:|---:|
| POST /api/v1/consumer-events [taxon_click] | 3,057 | 9 ms | 33 ms | 75 ms | 0 |
| POST /api/v1/recommendations/taxon | 3,057 | 8 ms | 20 ms | 31 ms | 0 |
| POST /api/v1/events [view_product] | 2,549 | 8 ms | 32 ms | 67 ms | 0 |
| POST /api/v1/recommendations/product | 2,549 | 7 ms | 18 ms | 35 ms | 0 |
| POST /api/v1/events [cart-events add] | 1,422 | 9 ms | 33 ms | 85 ms | 0 |
| POST /api/v1/recommendations/basket | 1,422 | 14 ms | 26 ms | 33 ms | 0 |
| POST /api/v1/feed | 1,025 | 14 ms | 28 ms | 45 ms | 0 |
| GET  /api/v1/health | 500 | 1 ms | 13 ms | 66 ms | 0 |
| POST /api/v1/events [order-events complete] | 499 | 8 ms | 34 ms | 110 ms | 0 |

### Aggregate

| Metric | Value |
|---|---|
| Total requests | **16,080** |
| Total failures | **0** |
| Failure rate | **0.00 %** |
| Sustained RPS | **~274 req/s** |
| p50 latency | **9 ms** |
| p95 latency | **28 ms** |
| p99 latency | **52 ms** |
| Max observed latency | 185 ms |

### Verdict

✅ **PASS** — Zero failures across 16,080 requests under 50 concurrent users.  
All placement endpoints (taxon, product, basket, feed) served under **35 ms p99**.  
The engine comfortably handles the simulated peak-hour traffic mix at **274 req/s** sustained throughput.


In [56]:
import subprocess, sys, re, json

# ── Run stress test and capture output ────────────────────────────────────────
# Edit --users / --duration to change the load profile
result = subprocess.run(
    [sys.executable, "tests/stress_test.py", "--users", "200", "--spawn-rate", "100", "--duration", "60"],
    capture_output=True, text=True,
    cwd="/workspaces/marketplace-stream-data-recommendation-engine",
)
out = result.stdout + result.stderr

# Extract summary block
summary_start = out.find("RESULTS SUMMARY")
print(out[summary_start:] if summary_start != -1 else out[-3000:])


RESULTS SUMMARY
════════════════════════════════════════════════════════════
  /api/v1/consumer-events [taxon_click]              n=  5734 p50= 260ms  p95=  370ms  p99=  460ms  fail=0
  /api/v1/recommendations/taxon                      n=  5714 p50= 190ms  p95=  280ms  p99=  370ms  fail=0
  /api/v1/events [view_product]                      n=  4757 p50= 260ms  p95=  380ms  p99=  470ms  fail=0
  /api/v1/recommendations/product                    n=  4748 p50= 190ms  p95=  280ms  p99=  370ms  fail=0
  /api/v1/events [cart-events add]                   n=  2852 p50= 260ms  p95=  380ms  p99=  480ms  fail=0
  /api/v1/recommendations/basket                     n=  2848 p50= 200ms  p95=  290ms  p99=  370ms  fail=0
  /api/v1/feed                                       n=  1902 p50= 190ms  p95=  290ms  p99=  370ms  fail=0
  /api/v1/events [order-events complete]             n=   941 p50= 260ms  p95=  380ms  p99=  460ms  fail=0
  /api/v1/health                                     n=   928 p50= 